# 1.1. Install and import dependencies

In [1]:
!pip install poetry
!python3 -m pip install --upgrade pip setuptools wheel

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 7.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 7.6 MB/s  0:00:00
   ---------------------------------------- 0.0/5.8 MB ? eta -:--:--
   ---------- ----------------------------- 1.6/5.8 MB 8.6 MB/s eta 0:00:01
   ----------------------- ---------------- 3.4/5.8 MB 8.6 MB/s eta 0:00:01
   ------------------------------------ --- 5.2/5.8 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------- 5.8/5.8 MB 8.6 MB/s  0:00:00

   - --------------------------------------  1/31 [fastjsonschema]
   -- -------------------------------------  2/31 [distlib]
   --- ------------------------------------  3/31 [zstandard]
   ----- ----------------------------------  4/31 [tomlkit]
   ------ ---------------------------------  5/31 [rapidfuzz]
   ------ ------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.


In [2]:
pyproject_content = """[tool.poetry]
name = "lab_1_example"
description = "Training pipeline for deep learning model"
authors = ["Name Surname <name.surname@gmail.com>"]
version = "0.01"

[tool.poetry.dependencies]
python = "~3.10"
torch = { version = "2.1.0+cu118", source = "pytorch" }
torchvision = { version = "0.16.0+cu118", source = "pytorch" }
tqdm = "4.64.1"
matplotlib = "3.6.3"
numpy = "1.22.4"
pyyaml = "6.0"
scipy = "1.13.0rc1"
pandas = ">2.0"

[tool.poetry.dev-dependencies]
mypy = "0.991"
ruff = "0.0.254"
black = "23.1.0"
isort = "5.12.0"

[build-system]
requires = ["poetry-core>=1.0.0"]
build-backend = "poetry.core.masonry.api"

[[tool.poetry.source]]
name = "pytorch"
priority = "supplemental"
url = "https://download.pytorch.org/whl/cu118" """

# Write the content to pyproject.toml
with open("pyproject.toml", "w") as file:
    file.write(pyproject_content)

# Step 2: Install dependencies using pip
!poetry export --without-hashes --output requirements.txt
!pip install -r requirements.txt

The requested command export does not exist.

Documentation: https://python-poetry.org/docs/cli/

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
import zipfile
import tarfile
import os
import logging
import requests

from pathlib import Path
from typing import Optional, Tuple, Dict, Any, Union

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_image
from torchvision import transforms

import scipy.io as sio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# hack to fix logging in colab https://stackoverflow.com/a/74121821/11709937
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO)

# 1.2 Data download

In [ ]:
def download_and_extract(url: str, save_dir: str, filename: Optional[str] = None) -> str:
    """
    Downloads a file from the given URL and extracts it if it is an archive.

    Args:
    - url (str): The URL of the file to download.
    - save_dir (str): Directory where the file will be saved and extracted.
    - filename (Optional[str]): Optional filename to save the file with. If None, uses the filename from the URL.

    Returns:
    - str: The full path to the directory containing the extracted files or the downloaded file.
    """
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    if filename is None:
        filename = url.split("/")[-1]

    file_path = save_path / filename
    result_path = str(file_path)

    # Check if file already exists
    if file_path.exists():
        logging.info(f"File '{filename}' already exists in '{save_dir}'. Skipping download.")
    else:
        logging.info(f"Downloading file '{filename}' from '{url}'...")
        try:
            response = requests.get(url)
            response.raise_for_status()
            with open(file_path, 'wb') as f:
                f.write(response.content)
            logging.info(f"Download successful. File saved to: '{file_path}'")

            # Extract the file if it's an archive
            if filename.endswith(".zip"):
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            elif (
                  filename.endswith(".tar.gz")
                  or filename.endswith(".tgz")
                  or filename.endswith(".gz")
                ):
                with tarfile.open(file_path, 'r:gz') as tar_ref:
                    tar_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            elif filename.endswith(".tar"):
                with tarfile.open(file_path, 'r') as tar_ref:
                    tar_ref.extractall(save_path)
                file_path.unlink()
                result_path = str(save_path)
            else:
                logging.info(f"File '{filename}' is not an archive. No extraction needed.")

        except requests.RequestException as e:
            logging.error(f"Error downloading file from {url}: {str(e)}")
            raise
        except (zipfile.BadZipFile, tarfile.ReadError) as e:
            logging.error(f"Error extracting file '{filename}': {str(e)}")
            raise

    return result_path

# 1.3 Data ingestion

In [ ]:
def train_test_split(
    data: pd.DataFrame,
    test_size: Union[float, int] = 0.25,
    random_state: Union[int, None] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split arrays or matrices into random train and test subsets.

    Parameters:
    X :  pd.DataFrame
         The input data to split.
    test_size : float, int, or None, optional (default=0.25)
        If float, should be between 0.0 and 1.0 and represent the proportion of the dataset to include in the test split.
        If int, represents the absolute number of test samples.
        If None, the value is set to the complement of the train size.
    random_state : int or None, optional (default=None)
        If int, random_state is the seed used by the random number generator;
        If None, the random number generator is the RandomState instance used by np.random.

    Returns:
    Tuple containing:
        - data_train: pd.DataFrame
        - data_test: pd.DataFrame
    """
    if random_state is not None:
        np.random.seed(random_state)

    n_samples = len(data)
    if isinstance(test_size, float):
        test_size = int(n_samples * test_size)

    indices = np.random.permutation(n_samples)
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    data_train = data.iloc[train_indices]
    data_test = data.iloc[test_indices]

    return data_train, data_test


def load_labels(labels_path: str) -> pd.DataFrame:
    """
    Load labels from a MAT file.

    Args:
    - labels_path (str): Path to the labels MAT file.

    Returns:
    - pd.DataFrame: DataFrame containing the labels.
    """
    labels_mat = sio.loadmat('/content/data/labels/imagelabels.mat')
    labels_df = pd.DataFrame({"label": labels_mat["labels"][0]})

    return labels_df

def find_add_images_to_labels(images_dir: str, labels: pd.DataFrame, image_ext: str="jpg") -> pd.DataFrame:
    image_paths = sorted(
        [str(image_path.absolute()) for image_path in Path(images_dir).rglob(f"*.{image_ext}")]
    )
    if len(image_paths) != len(labels):
        logging.error(
            f"Found {len(image_paths)} image_paths but "
            f"{len(labels)} labels were provided, cannot continue."
        )
        raise ValueError

    labels_w_image_paths = labels.copy(deep=True)
    labels_w_image_paths["image_path"] = image_paths
    return labels_w_image_paths

def process_data(images_dir: str, labels_path: str, config: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Process the data by loading labels, validating data, splitting into train-validation-test sets,
    and optionally saving the splits.

    Args:
    - images_dir (str): Directory containing the images.
    - labels_path (str): Path to the labels MAT file.
    - config (Dict[str, Any]): Configuration dictionary containing parameters for data splitting and paths.

    Returns:
    - Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: Training, validation, and testing DataFrames.
    """
    # Load labels
    labels_df = load_labels(labels_path)

    # Validate data (if needed, adjust based on your specific validation logic)
    valid_labels_df = find_add_images_to_labels(images_dir, labels_df)

    # Split data into train and test
    train_df, test_df = train_test_split(valid_labels_df, test_size=config.get('test_size', 0.2), random_state=config.get('random_state', 42))

    # Further split train_df into train and validation
    train_df, val_df = train_test_split(train_df, test_size=config.get('val_size', 0.2), random_state=config.get('random_state', 42))
    logging.info(f"Prepared 3 data splits: train, size: {len(train_df)}, val: {len(val_df)}, test: {len(val_df)}")
    return train_df, val_df, test_df

# 1.4 Training loop

## 1.4.1 Data loaders

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform: Optional[Any] = None) -> None:
        self.dataframe = dataframe
        self.transform = transform
        self.transform_default = transforms.Compose([
            transforms.Resize([28, 28]),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        # replace with your own implementation if needed
        img_path = self.dataframe.iloc[idx]["image_path"]
        image = read_image(img_path).float() / 255.0
        # labels in this dataset start from 1, but we need to start from 0
        label = int(self.dataframe.iloc[idx]["label"]) - 1

        if self.transform:
            image = self.transform(image)
        else:
            image = self.transform_default(image)

        return image, label

def create_data_loader(
    images_dir: str,
    df: pd.DataFrame,
    config: Dict[str, Any]
) -> DataLoader:
    """
    Create a data loader for a dataset.


    Args:
    - images_dir (str): Directory containing the images.
    - df (pd.DataFrame): DataFrame containing the dataset.
    - config (Dict[str, Any]): Configuration dictionary containing parameters for data loading.

    Returns:
    - DataLoader: DataLoader for the dataset.
    """
    transform: Optional[Any] = config.get('transform', None)
    batch_size: int = config.get('batch_size', 32)
    num_workers: int = config.get('num_workers', 2)

    dataset = ImageDataset(df, transform=transform)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    return data_loader

## 1.4.2. Model, loss and optimizers

In [ ]:
    # Model Definition
    class SimpleNN(nn.Module):
        def __init__(self, n_classes: int) -> None:
            super(SimpleNN, self).__init__()
            self.fc1 = nn.Linear(784*3, 128)  # Example for an input size of 784*3 (e.g., flattened 28x28 image) and hidden layer with 128 units
            self.fc2 = nn.Linear(128, n_classes)   # Output layer with 10 units (e.g., 10 classes for classification)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = torch.flatten(x, 1)  # Flatten the input tensor except for the batch dimension
            x = torch.relu(self.fc1(x))
            x = self.fc2(x)
            return x

## 1.4.3. Training loop function

In [ ]:
def train_model(
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        loss_function: nn.Module,
        optimizer: optim.Optimizer,
        num_epochs: int,
        device: torch.device,
        save_path: Path = Path("best_model.pth")
        ) -> Path:

    model.to(device)
    best_val_loss: float = float('inf')
    best_model_path: Path = Path()

    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        for batch_inputs, batch_targets in train_loader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

            # Forward pass
            outputs: torch.Tensor = model(batch_inputs)
            loss: torch.Tensor = loss_function(outputs, batch_targets)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        logging.info(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")


        # Validation step
        model.eval()  # Set the model to evaluation mode

        with torch.no_grad():

            val_loss: float = 0.0
            for val_inputs, val_targets in val_loader:

                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
                val_outputs: torch.Tensor = model(val_inputs)
                val_loss += loss_function(val_outputs, val_targets).item()

            val_loss /= len(val_loader)
            logging.info(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss:.4f}")

            # Save the best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_path = save_path
                torch.save(model.state_dict(), best_model_path)

                logging.info(f"Best model saved with validation loss: {best_val_loss:.4f}")

    logging.info("Training complete.")

    return best_model_path

# 1.5 Test loop

In [ ]:
def test_model(
        model: nn.Module,
        test_loader: DataLoader,
        loss_function: nn.Module,
        device: torch.device
        ) -> float:
    """
    Test a trained model on a test dataset and compute test metrics.


    Args:
    - model (nn.Module): Trained PyTorch model.
    - test_loader (DataLoader): DataLoader for the test dataset.
    - loss_function (nn.Module): Loss function for computing the loss.
    - device (torch.device): Device (CPU or GPU) on which to run the evaluation.

    Returns:
    - float: Test loss.
    """

    model.eval()  # Set the model to evaluation mode
    test_loss = 0.0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = loss_function(outputs, targets)
            test_loss += loss.item()

    test_loss /= len(test_loader)
    logging.info(f"Test Loss: {test_loss:.4f}")

    return test_loss

# 1.6. Put everything together

In [ ]:
def main():
    # Configuration
    config = {
        'test_size': 0.2,
        'val_size': 0.2,
        'random_state': 42,
        "lr": 0.001
        # Add other configuration parameters as needed
    }
    # Step 1: Download data
    images_url = 'https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz'
    labels_url = 'https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat'
    images_dir = download_and_extract(images_url, '/content/data/images')
    labels_path = download_and_extract(labels_url, '/content/data/labels')

    # Step 2: Process data
    train_df, val_df, test_df = process_data(images_dir, labels_path, config)

    # Step 3: Create data loaders
    train_loader = create_data_loader(images_dir, train_df, config)
    val_loader = create_data_loader(images_dir, val_df, config)
    test_loader = create_data_loader(images_dir, test_df, config)

    # Step 4: Define model, loss function, optimizer
    model = SimpleNN(n_classes=102).to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=config["lr"])

    # Step 5: Train model
    best_model_path = train_model(model, train_loader, val_loader, loss_function, optimizer, num_epochs=10, device=device)

    # Step 6: Test model
    test_model(model, test_loader, loss_function, device)

In [ ]:
main()

INFO:root:Downloading file '102flowers.tgz' from 'https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz'...
INFO:root:Download successful. File saved to: '/content/data/images/102flowers.tgz'
INFO:root:File 'imagelabels.mat' already exists in '/content/data/labels'. Skipping download.
INFO:root:Prepared 3 data splits: train, size: 5242, val: 1310, test: 1310
INFO:root:Epoch 1/10, Loss: 4.4943
INFO:root:Epoch 1/10, Validation Loss: 4.5283
INFO:root:Best model saved with validation loss: 4.5283


KeyboardInterrupt: 